In [ ]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/processed/uber_clean.csv")

df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"])

In [3]:
df.head()

,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06+00:00,-73.999817,40.738354,-73.999512,40.723217,1
1,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56+00:00,-73.994355,40.728225,-73.994710,40.750325,1
2,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00+00:00,-74.005043,40.740770,-73.962565,40.772647,1
3,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21+00:00,-73.976124,40.790844,-73.965316,40.803349,3
4,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00+00:00,-73.925023,40.744085,-73.973082,40.761247,5


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 195333 entries, 0 to 195332
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype              
---  ------             --------------   -----              
 0   key                195333 non-null  str                
 1   fare_amount        195333 non-null  float64            
 2   pickup_datetime    195333 non-null  datetime64[us, UTC]
 3   pickup_longitude   195333 non-null  float64            
 4   pickup_latitude    195333 non-null  float64            
 5   dropoff_longitude  195333 non-null  float64            
 6   dropoff_latitude   195333 non-null  float64            
 7   passenger_count    195333 non-null  int64              
dtypes: datetime64[us, UTC](1), float64(5), int64(1), str(1)
memory usage: 17.1 MB


# Feature 1
## Temporal Features
The pickup timestamp contains valuable information about ride behaviour.
Several features will be extracted from the timestamp.

In [27]:
df["pickup_hour"] = df["pickup_datetime"].dt.hour

In [6]:
df["pickup_day"] = df["pickup_datetime"].dt.day_name()

In [7]:
df["day_of_week"] = df["pickup_datetime"].dt.dayofweek

In [8]:
df["pickup_month"] = df["pickup_datetime"].dt.month

In [9]:
df["pickup_year"] = df["pickup_datetime"].dt.year

In [10]:
df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)

In [11]:
df["is_peak_hour"] = (
    (
        (df["pickup_hour"] >= 7) &
        (df["pickup_hour"] <= 9)
    )
    |
    (
        (df["pickup_hour"] >= 16) &
        (df["pickup_hour"] <= 19)
    )
).astype(int)

In [12]:
df.head()

,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,pickup_hour,pickup_day,day_of_week,pickup_month,pickup_year,is_weekend,is_peak_hour
0,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06+00:00,-73.999817,40.738354,-73.999512,40.723217,1,19,Thursday,3,5,2015,0,1
1,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56+00:00,-73.994355,40.728225,-73.994710,40.750325,1,20,Friday,4,7,2009,0,0
2,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00+00:00,-74.005043,40.740770,-73.962565,40.772647,1,21,Monday,0,8,2009,0,0
3,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21+00:00,-73.976124,40.790844,-73.965316,40.803349,3,8,Friday,4,6,2009,0,1
4,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00+00:00,-73.925023,40.744085,-73.973082,40.761247,5,17,Thursday,3,8,2014,0,1


# Feature 2
## Trip Distance
The Haversine Formula computes the great-circle distance between two points on the Earth's surface.
This distance is expected to be one of the strongest predictors of fare amount.

In [13]:
from math import radians
from math import sin
from math import cos
from math import sqrt
from math import atan2

In [14]:
R = 6371  # Earth's radius in kilometers

lat1 = np.radians(df["pickup_latitude"])
lon1 = np.radians(df["pickup_longitude"])

lat2 = np.radians(df["dropoff_latitude"])
lon2 = np.radians(df["dropoff_longitude"])

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat1)
    * np.cos(lat2)
    * np.sin(dlon / 2) ** 2
)

c = 2 * np.arctan2(
    np.sqrt(a),
    np.sqrt(1 - a)
)

df["trip_distance_km"] = R * c

In [15]:
df["trip_distance_km"].describe()

count    195333.000000
mean          5.153867
std         104.352154
min           0.000000
25%           1.255851
50%           2.157924
75%           3.911449
max        8708.233063
Name: trip_distance_km, dtype: float64

In [16]:
df["trip_distance_km"].isnull().sum()

np.int64(0)

In [17]:
(df["trip_distance_km"]<0).sum()

np.int64(0)

In [20]:
import plotly.express as px
fig = px.histogram(
    df,
    x="trip_distance_km",
    nbins=60,
    title="Distribution of Trip Distance",
    template="plotly_white"
)
fig.show()

### Observation
- Most trips are relatively short.
- Long-distance trips are uncommon.
- The distribution is right-skewed.

### Business Insight
Trip distance is expected to have a strong positive relationship with fare amount, making it one of the most important predictive features.

In [21]:
sample_df = df.sample(10000, random_state=42)

fig = px.scatter(
    sample_df,
    x="trip_distance_km",
    y="fare_amount",
    opacity=0.5,
    title="Trip Distance vs Fare Amount",
    template="plotly_white"
)

fig.update_layout(
    title_x=0.5,
    xaxis_title="Distance (km)",
    yaxis_title="Fare Amount ($)",
    height=600
)

fig.show()

### Observation
- Fare generally increases with trip distance.
- The relationship appears positively correlated.
- Some variability exists due to factors such as traffic, route selection, tolls, and surge pricing.

### Business Insight
Trip distance is expected to become the most influential feature during model training.

In [ ]:
corr_df = df[
    [
        "fare_amount",
        "trip_distance_km",
        "pickup_hour",
        "pickup_month",
        "passenger_count",
        "is_weekend",
        "is_peak_hour"
    ]
]

In [23]:
corr = corr_df.corr(numeric_only=True)

In [24]:
fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    title="Feature Correlation Matrix"
)

fig.update_layout(
    title_x=0.5
)

fig.show()

In [25]:
df.to_csv(
    "../data/processed/uber_feature_engineered.csv",
    index=False
)